<a href="https://colab.research.google.com/github/Tobias0809/smolagents-reporting-agent/blob/main/agentic_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
!pip install smolagents[toolkit] pytz -q
from smolagents import CodeAgent, DuckDuckGoSearchTool, FinalAnswerTool, InferenceClientModel, load_tool, tool
import datetime
import requests
import pytz
import yaml
import random
from datetime import date, timedelta
import sqlite3

from google.colab import userdata

In [2]:
# creating a database with synthetic data called sales.db
#2


DB_PATH = "sales.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS sales")
cur.execute("""
CREATE TABLE sales (
    sale_date   TEXT NOT NULL,
    region      TEXT NOT NULL,
    category    TEXT NOT NULL,
    product     TEXT NOT NULL,
    units       INTEGER NOT NULL,
    revenue     REAL NOT NULL
)
""")

random.seed(42)

catalog = {
    "Industrial Sensors": ["SN-100", "SN-220", "SN-350"],
    "Control Units":      ["CU-10", "CU-40"],
    "Spare Parts":        ["SP-01", "SP-02", "SP-07"],
    "Service Contracts":  ["SVC-BASIC", "SVC-PLUS"],
}
regions = ["DACH", "Nordics", "Benelux", "Southern Europe"]

# standard price via category
base_price = {
    "Industrial Sensors": 420.0,
    "Control Units": 1350.0,
    "Spare Parts": 65.0,
    "Service Contracts": 2200.0,
}

# annual trend (for data exploration)
yearly_trend = {
    "Industrial Sensors": 1.12,
    "Control Units": 1.03,
    "Spare Parts": 0.88,
    "Service Contracts": 1.25,
}

start = date(2023, 1, 1)
end = date(2025, 12, 31)

rows = []
current = start
while current <= end:
    for category, products in catalog.items():
        for product in products:
            for region in regions:
                if random.random() < 0.55:
                    continue

                years_passed = (current - start).days / 365.0
                trend = yearly_trend[category] ** years_passed

                # "small" saisonality
                season = 1.0 + 0.18 * (1 if current.month in (3, 4, 9, 10, 11) else -1) * random.random()

                units = max(1, int(random.gauss(14, 6) * trend * season))
                price = base_price[category] * random.uniform(0.92, 1.08)
                revenue = round(units * price, 2)

                rows.append((current.isoformat(), region, category, product, units, revenue))

    current += timedelta(days=1)

cur.executemany(
    "INSERT INTO sales (sale_date, region, category, product, units, revenue) VALUES (?, ?, ?, ?, ?, ?)",
    rows,
)
conn.commit()

print(f"rows created: {len(rows)}")
min_date, max_date = cur.execute(
    "SELECT MIN(sale_date), MAX(sale_date) FROM sales"
).fetchone()

print(f"Data available from {min_date} to {max_date}")
conn.close()




rows created: 19700
Data available from 2023-01-01 to 2025-12-31


In [3]:
# ==========================================================
# implementing the tools (tools are what let the agent act beyond text,
# the model only decides which tool to call, the code below does the actual work)
'''
get_schema = returns the structure of the database, the columns and the distinct
values for category, product and region, so the agent knows what exists before
it queries anything instead of guessing names or date ranges

aggregate_sales = sums revenue or units over a given time period and groups the
result by one dimension, which covers the standard group by question a business
user would otherwise ask a report for

compare_periods = takes one dimension value and returns the absolute and relative
change between two time periods, done in code because language models are not
reliable at arithmetic
'''

# ==========================================================
#3 tools
DB_PATH = "sales.db"

ALLOWED_METRICS = {"revenue": "SUM(revenue)", "units": "SUM(units)"} #only metrics the agent can retrieve data from. more sphisitcated user queries like annual recurring revenue (ARR) can not be answered
ALLOWED_DIMENSIONS = {"category", "product", "region"} # columns the agent is allowed to retrieve from, limitation is ,e.g., for data privacy reasons


def _query(sql: str, params: tuple = ()) -> list:

    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    try:
        return conn.execute(sql, params).fetchall()
    finally:
        conn.close()


@tool
def get_schema() -> str:
    """Returns the structure of the sales database, including all columns and the
    distinct values available for the dimension columns. Call this first to find
    out which categories, products and regions exist before running any aggregation.
    """
    lines = [
        "Table: sales",
        "Columns: sale_date (TEXT, format YYYY-MM-DD), region (TEXT), "
        "category (TEXT), product (TEXT), units (INTEGER), revenue (REAL)",
    ]
    for dim in ("category", "product", "region"):
        values = [r[0] for r in _query(f"SELECT DISTINCT {dim} FROM sales ORDER BY {dim}")]
        lines.append(f"Distinct {dim} values: {', '.join(values)}")

    date_range = _query("SELECT MIN(sale_date), MAX(sale_date) FROM sales")[0]
    lines.append(f"Data available from {date_range[0]} to {date_range[1]}")
    return "\n".join(lines)


@tool
def aggregate_sales(metric: str, group_by: str, start_date: str, end_date: str) -> str:
    """Aggregates sales figures over a time period and groups them by one dimension.

    Args:
        metric: What to sum up. Either 'revenue' or 'units'.
        group_by: The dimension to group by. One of 'category', 'product' or 'region'.
        start_date: First day of the period, format YYYY-MM-DD.
        end_date: Last day of the period, format YYYY-MM-DD.
    """
    if metric not in ALLOWED_METRICS:
        return f"Error: metric must be one of {sorted(ALLOWED_METRICS)}. Received '{metric}'."
    if group_by not in ALLOWED_DIMENSIONS:
        return f"Error: group_by must be one of {sorted(ALLOWED_DIMENSIONS)}. Received '{group_by}'."

    sql = f"""
        SELECT {group_by}, {ALLOWED_METRICS[metric]}
        FROM sales
        WHERE sale_date BETWEEN ? AND ?
        GROUP BY {group_by}
        ORDER BY 2 DESC
    """
    rows = _query(sql, (start_date, end_date))
    if not rows:
        return f"No data found between {start_date} and {end_date}."

    lines = [f"{metric} by {group_by}, {start_date} to {end_date}:"]
    for name, value in rows:
        lines.append(f"  {name}: {value:,.2f}")
    return "\n".join(lines)


@tool
def compare_periods(metric: str, dimension: str, value: str,
                    period_a_start: str, period_a_end: str,
                    period_b_start: str, period_b_end: str) -> str:
    """Compares one metric for a single dimension value between two time periods and
    returns the absolute and relative change.

    Args:
        metric: What to sum up. Either 'revenue' or 'units'.
        dimension: The column to filter on. One of 'category', 'product' or 'region'.
        value: The value to filter for, for example 'Spare Parts'.
        period_a_start: First day of the earlier period, format YYYY-MM-DD.
        period_a_end: Last day of the earlier period, format YYYY-MM-DD.
        period_b_start: First day of the later period, format YYYY-MM-DD.
        period_b_end: Last day of the later period, format YYYY-MM-DD.
    """
    if metric not in ALLOWED_METRICS:
        return f"Error: metric must be one of {sorted(ALLOWED_METRICS)}. Received '{metric}'."
    if dimension not in ALLOWED_DIMENSIONS:
        return f"Error: dimension must be one of {sorted(ALLOWED_DIMENSIONS)}. Received '{dimension}'."

    sql = f"""
        SELECT {ALLOWED_METRICS[metric]}
        FROM sales
        WHERE {dimension} = ? AND sale_date BETWEEN ? AND ?
    """
    a = _query(sql, (value, period_a_start, period_a_end))[0][0] or 0
    b = _query(sql, (value, period_b_start, period_b_end))[0][0] or 0

    if a == 0:
        return f"No data for {value} in the first period, cannot compute a change."

    delta = b - a
    pct = delta / a * 100

    return (
        f"{metric} for {dimension} = {value}\n"
        f"  {period_a_start} to {period_a_end}: {a:,.2f}\n"
        f"  {period_b_start} to {period_b_end}: {b:,.2f}\n"
        f"  Change: {delta:,.2f} ({pct:+.1f} percent)"
    )



#test without agent to be able to distinguish between tool and agent mistakes
print(get_schema())
print()
print(aggregate_sales("revenue", "category", "2025-01-01", "2025-12-31"))


Table: sales
Columns: sale_date (TEXT, format YYYY-MM-DD), region (TEXT), category (TEXT), product (TEXT), units (INTEGER), revenue (REAL)
Distinct category values: Control Units, Industrial Sensors, Service Contracts, Spare Parts
Distinct product values: CU-10, CU-40, SN-100, SN-220, SN-350, SP-01, SP-02, SP-07, SVC-BASIC, SVC-PLUS
Distinct region values: Benelux, DACH, Nordics, Southern Europe
Data available from 2023-01-01 to 2025-12-31

revenue by category, 2025-01-01 to 2025-12-31:
  Service Contracts: 69,935,767.59
  Control Units: 25,805,879.49
  Industrial Sensors: 14,752,894.06
  Spare Parts: 1,201,418.55


In [17]:
# ==========================================================
# CELL 4  the agent
# ==========================================================
# assembling the agent (the agent is the loop, the model is the decision inside
# that loop, the tools are the execution, nothing runs without this wiring)
'''
InferenceClientModel = the connection to the LLM through the Hugging Face inference
API, the model itself runs on their servers, this object only sends the prompt and
returns the generated text

CodeAgent = the loop that keeps calling the model, parses the action it emits,
executes the matching tool and feeds the real result back as an observation until
a final answer is reached

FinalAnswerTool = the stop signal, the agent calls it to leave the loop, without it
the run continues until max_steps is hit
'''

import os
from smolagents import CodeAgent, InferenceClientModel, FinalAnswerTool

# in Colab: key icon on the left, add a secret named HF_TOKEN, enable notebook access
# keeping the token out of the code so it never ends up in the repository
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",  # a code oriented model, it has to emit executable python as its action
    max_tokens=2096,                             # upper bound on the length of a single generation
    temperature=0.2,                             # low randomness, the agent should be predictable rather than creative
)

agent = CodeAgent(
    model=model,
    tools=[get_schema, aggregate_sales, compare_periods, FinalAnswerTool()],  # the agent only knows the tools listed here
    max_steps=8,          # hard limit on the loop, prevents an agent that never converges from running forever
    verbosity_level=2,    # prints thought, action and observation for every step, this is how the reasoning becomes auditable
)


In [19]:

# ==========================================================
# CELL 5  running the agent
# ==========================================================
# a question that cannot be answered in one call, the agent has to look up the
# schema, pull the figures for both years and only then draw a conclusion
antwort = agent.run(
    "Which product category declined the most in revenue in 2025 compared to 2024? "
    "Check the schema first, then give me the figures and a one sentence interpretation."
)
print(antwort)


# further questions worth trying
#
# agent.run("What was total revenue per region in the second half of 2025?")
# agent.run("Compare units sold for Industrial Sensors between 2023 and 2025.")
# agent.run("Which product performed best in the Nordics last year?")
#
# and one that is deliberately out of scope, to see how the agent handles a metric
# that does not exist in the whitelist
#
# agent.run("What was the gross margin per category in 2025?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Which product category declined the most in revenue in 2025 compared to 2024? Check the schema first, then give │
│ me the figures and a one sentence interpretation.                                                               │
│                                                                                                                 │
╰─ InferenceClientModel - Qwen/Qwen2.5-Coder-32B-Instruct ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to first check the schema of the sales database to understand the available categories, products,  
and regions. After that, I will use the `compare_periods` tool to get the revenue figures for each category in 2024
and 2025, and then determine which category declined the most.                                                     
                                                                                                                   
<code>                                                                                                             
schema = get_schema()                                                                                              
print(schema)                                                                                                      
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  schema = get_schema()                                                                                            
  print(schema)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Table: sales
Columns: sale_date (TEXT, format YYYY-MM-DD), region (TEXT), category (TEXT), product (TEXT), units (INTEGER), 
revenue (REAL)
Distinct category values: Control Units, Industrial Sensors, Service Contracts, Spare Parts
Distinct product values: CU-10, CU-40, SN-100, SN-220, SN-350, SP-01, SP-02, SP-07, SVC-BASIC, SVC-PLUS
Distinct region values: Benelux, DACH, Nordics, Southern Europe
Data available from 2023-01-01 to 2025-12-31

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: From the observations, I can see the revenue changes for each category. The category that declined the    
most in revenue from 2024 to 2025 is "Spare Parts" with a decline of 13.0 percent. I will now prepare a            
one-sentence interpretation of this finding.                                                                       
                                                                                                                   
<code>                                                                                                             
final_answer("The product category that declined the most in revenue in 2025 compared to 2024 is Spare Parts, with 
a decline of 13.0 percent.")                                                                                       
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The product category that declined the most in revenue in 2025 compared to 2024 is Spare Parts,    
  with a decline of 13.0 percent.")                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The product category that declined the most in revenue in 2025 compared to 2024 is Spare Parts, with 
a decline of 13.0 percent.

[Step 3: Duration 3.11 seconds| Input tokens: 8,815 | Output tokens: 380]

The product category that declined the most in revenue in 2025 compared to 2024 is Spare Parts, with a decline of 13.0 percent.


In [ ]:

# ==========================================================
# CELL 6  optional, clickable interface
# ==========================================================
# wraps the same agent in a chat UI, share=True exposes a public link that stays
# alive for 72 hours, useful for a live demo but not as a permanent reference
# from smolagents import GradioUI
# GradioUI(agent).launch(share=True)